In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import os

# =========================
# 📌 PATH
# =========================
base_path = "/kaggle/input/competitions/yzta-2026-datathon"

train = pd.read_csv(f"{base_path}/train.csv")
test = pd.read_csv(f"{base_path}/test_x.csv")

# =========================
# 📌 TARGET
# =========================
TARGET = "bilissel_performans_skoru"
id_col = "id"

# =========================
# 📌 NUMERIC COLUMNS
# =========================
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop([TARGET, id_col], errors='ignore')

# =========================
# 📌 FEATURE ENGINEERING
# =========================
for df in [train, test]:
    df['mean_val'] = df[num_cols].mean(axis=1)
    df['std_val'] = df[num_cols].std(axis=1)

# =========================
# 📌 NaN FIX (KRİTİK)
# =========================

# categorical NaN → string
for col in train.columns:
    if train[col].dtype == 'object':
        train[col] = train[col].fillna("missing")
        test[col] = test[col].fillna("missing")

# numeric NaN → median
for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(train[col].median())

# feature engineering sonrası oluşan NaN fix
train = train.fillna(0)
test = test.fillna(0)

# =========================
# 📌 DATASET
# =========================
X = train.drop(columns=[TARGET])
y = train[TARGET]

# sütun hizalama
test = test[X.columns]

cat_features = list(X.select_dtypes(include='object').columns)

# =========================
# 📌 MODEL
# =========================
model = CatBoostRegressor(
    iterations=2500,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=5,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=100,
    verbose=250
)

model.fit(X, y, cat_features=cat_features)

# =========================
# 📌 PREDICTION
# =========================
preds = model.predict(test)

# =========================
# 📌 SUBMISSION
# =========================
submission = pd.DataFrame({
    id_col: test[id_col],
    TARGET: preds
})

submission.to_csv("submission.csv", index=False)

print("✔ Submission hazır")

0:	learn: 2.1956337	total: 110ms	remaining: 4m 35s
250:	learn: 1.2138745	total: 8.89s	remaining: 1m 19s
500:	learn: 1.1918641	total: 17.2s	remaining: 1m 8s
750:	learn: 1.1776484	total: 25.5s	remaining: 59.4s
1000:	learn: 1.1654868	total: 33.6s	remaining: 50.4s
1250:	learn: 1.1543233	total: 41.7s	remaining: 41.6s
1500:	learn: 1.1436318	total: 49.7s	remaining: 33.1s
1750:	learn: 1.1317929	total: 57.9s	remaining: 24.8s
2000:	learn: 1.1199634	total: 1m 6s	remaining: 16.5s
2250:	learn: 1.1079789	total: 1m 14s	remaining: 8.27s
2499:	learn: 1.0958358	total: 1m 23s	remaining: 0us
✔ Submission hazır
